# Part 14 — Productivity-proxy validation + RF cross-check (RQ1 validation)

Validate the knowledge-based suitability against realized productivity and the realized niche: **Spearman ρ** of municipal mean suitability vs a MODIS MOD17 NPP proxy (cropland-restricted), the **Boyce index / AUC** of crop presence vs the suitability surface, an **ANOVA** of the proxy across zones, and a data-driven **random-forest** cross-check quantified by **Cohen's κ**. Engines: `src/external.py` + `src/metrics.py`.

**Caveats:** MOD17 is *vegetation primary productivity, not agronomic yield* (coarse surrogate); the in-stack NDVI integral is excluded as validator (it is a suitability input → circular), so MOD17 is a distinct product with a noted residual shared-signal caveat.

**Output:** validation tables (κ / ρ / Boyce / AUC / ANOVA). **DoD:** RQ1 validated.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import external, metrics
import pandas as pd

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Build the MOD17 NPP productivity proxy (250 m multi-year mean, fills removed)

In [ ]:
proxy = external.mod17_proxy(aoi)
utils.range_report(proxy, aoi)

### Spearman ρ — municipal mean suitability vs MOD17 (cropland-restricted)

In [ ]:
suit = ee.Image(utils.asset_id(project, 'suit_present'))
realized = external.realized_features(aoi)
crop = external.cropland_mask(realized)
muni = external.municipal_fc()
segs = list(utils.cfg('segments')['segments'])
agg = suit.select([f'suit_{s}' for s in segs]).addBands(proxy).updateMask(crop)
feats = external.municipal_means(muni, agg).select([f'suit_{s}' for s in segs] + ['mod17_npp'], None, False).getInfo()['features']
df = pd.DataFrame([f['properties'] for f in feats]).dropna()
for s in segs:
    r = metrics.spearman(df[f'suit_{s}'], df['mod17_npp'])
    print(f"{s:14s} rho={r['rho']:+.3f}  p={r['p']:.1e}  n={r['n']}")

### Repaired crop validator — WITHIN-crop suitability→season-GPP gradient (2026-07-14)
The annual NPP proxy above is confounded across land cover (positively tied to standing biomass, negatively to bare-fallow crops — the spurious soybean −0.34). The repair: **growing-season GPP** (`external.season_gpp`, MOD17A2HGF integrated over the Oct–Mar canopy window) evaluated **only on each crop's own realized pixels**, correlated against that crop's suitability (`metrics.within_crop_gradient`). This validates *how much* crops produce; the Boyce/AUC below validate *where*. Sign should be **non-negative** for the extensive crops. (Residual shared signal with the in-stack `phen_integral` remains — an IBGE-yield validator is the future upgrade.)

In [ ]:
gpp = external.season_gpp(aoi)  # sugarcane: pass months=list(range(1,13))
for crop in ['soybean', 'sugarcane', 'other_crops']:
    m = realized.select('rl_role').eq(external.ROLE_CODES[crop])
    smp = (suit.select(f'suit_{crop}').addBands(gpp).updateMask(m)
           .sample(region=aoi, scale=250, numPixels=8000, seed=7, dropNulls=True)
           .getInfo()['features'])
    g = pd.DataFrame([f['properties'] for f in smp])
    r = metrics.within_crop_gradient(g[f'suit_{crop}'], g['season_gpp'])
    print(f"{crop:12s} within-crop rho={r['rho']:+.3f} p={r['p']:.1e} n={r['n']} monotonic={r['monotonic']}")

### Boyce index / AUC — soybean presence (MapBiomas) vs the suitability surface

In [ ]:
smp = (suit.select('suit_soybean').addBands(realized.select('rl_role'))
       .sample(region=aoi, scale=250, numPixels=20000, seed=1, dropNulls=True)
       .getInfo()['features'])
sdf = pd.DataFrame([f['properties'] for f in smp])
pres = sdf[sdf.rl_role == external.ROLE_CODES['soybean']]['suit_soybean']
print('soybean Boyce index:', round(metrics.continuous_boyce(pres, sdf['suit_soybean'])['boyce'], 3))
sdf['is_soy'] = (sdf.rl_role == external.ROLE_CODES['soybean']).astype(int)
print('soybean AUC:', round(metrics.auc(sdf['is_soy'], sdf['suit_soybean'])['auc'], 3))

### ANOVA — MOD17 proxy across biophysical zones

In [ ]:
zones = ee.Image(utils.asset_id(project, 'zones_present')).rename('zone')
zs = (proxy.addBands(zones).sample(region=aoi, scale=1000, numPixels=15000, seed=2, dropNulls=True)
      .getInfo()['features'])
zdf = pd.DataFrame([f['properties'] for f in zs])
groups = {int(z): g['mod17_npp'].values for z, g in zdf.groupby('zone')}
print(metrics.anova(groups))

### Data-driven RF cross-check — Cohen's κ vs the knowledge-based map
Sample soybean presence/background on the z-stack, train `smileRandomForest` in PROBABILITY mode, threshold at 0.5, and compare to the knowledge-based S2+ map. Disagreements inform one `segments.yaml` recalibration pass.

In [ ]:
z = ee.Image(utils.asset_id(project, 'feature_stack_250m_z'))
lbl = realized.select('rl_role').eq(external.ROLE_CODES['soybean']).rename('presence')
train = z.addBands(lbl).stratifiedSample(numPoints=3000, classBand='presence',
        region=aoi, scale=250, seed=3, dropNulls=True)
rf = (ee.Classifier.smileRandomForest(100).setOutputMode('PROBABILITY')
      .train(train, 'presence', z.bandNames()))
rf_cls = z.classify(rf).gte(0.5).rename('rf_cls')
cmp = rf_cls.addBands(suit.select('class_soybean').gte(2).rename('kb_cls'))
cs = cmp.sample(region=aoi, scale=250, numPixels=10000, seed=4, dropNulls=True).getInfo()['features']
cdf = pd.DataFrame([f['properties'] for f in cs])
print('soybean RF-vs-knowledge Cohen κ:', metrics.cohen_kappa(cdf['rf_cls'], cdf['kb_cls']))